# 🎙️ Treinar voz com IA — Especial Caçamba dos Cria

Transforma gravações (MP3, WAV, vídeo…) de uma pessoa numa **voz com IA** que dá pra usar ao vivo no app.

**Como usar:** vá em *Ambiente de execução → Alterar o tipo de ambiente* e escolha **GPU T4**. Depois clique no ▶️ de cada passo, de cima pra baixo.

- Quanto mais áudio limpo, melhor: **10 a 30 minutos** de fala é o ideal (dá pra fazer com 3–5 min, fica pior).
- Só a voz da pessoa. Se tiver música ou barulho, marque *separar voz* no passo 3.
- No final baixa um arquivo `.pth` pequeno (~55 MB). Abra o **Estúdio de Vozes**, clique em *Carregar modelo*, escolha o `.pth`, teste e publique.

⚠️ Use só vozes que você tem direito de usar. Não use pra enganar ninguém se passando por outra pessoa.

## 1. Instalar (uns 5 minutos)

In [7]:
import os; os.environ["MPLBACKEND"] = "Agg"

In [1]:
# @title Instalar o Applio
import os, sys, subprocess
APPLIO = "/content/Applio"
PY = f"{APPLIO}/.venv/bin/python"
UV = f"{sys.executable} -m uv"
!{sys.executable} -m pip install -q uv
if not os.path.exists(f"{APPLIO}/assets/config.json"):
    !rm -rf /content/Applio
    %cd /content
    !git -c advice.detachedHead=false clone --depth 1 --branch 3.7.1 https://github.com/IAHispano/Applio
    %cd /content/Applio
    !apt -qq update -y > /dev/null && apt -qq install -y portaudio19-dev ffmpeg > /dev/null
    !{UV} venv .venv --python 3.12 --seed -q
    !{UV} pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python {PY}
    !{PY} core.py prerequisites --models --pretraineds-hifigan --exe
    !cp assets/config_template.json assets/config.json
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
ok = subprocess.run([PY, "-c", "import torch, faiss, librosa"], capture_output=True).returncode == 0 if os.path.exists(PY) else False
if not ok:
    !rm -f {APPLIO}/assets/config.json
    raise SystemExit("❌ A instalação falhou. Veja as mensagens acima e rode este passo de novo.")
print("✅ Pronto!" if gpu else "⚠️ Sem GPU: vá em Ambiente de execução → Alterar o tipo de ambiente → GPU T4 e rode de novo.")
if gpu: print("Placa:", gpu)

/content
Cloning into 'Applio'...
remote: Enumerating objects: 267, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 267 (delta 21), reused 137 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (267/267), 17.73 MiB | 18.82 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/Applio


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


No executables needed

Prerequisites downloaded and installed successfully.
✅ Pronto!
Placa: Tesla T4


## 2. Mandar os áudios

In [2]:
# @title Nome da voz e envio dos arquivos
# @markdown Nome sem espaço nem acento (ex: `lula`, `faustao`).
nome_da_voz = "Lula"  # @param {type:"string"}
import re, os, shutil, subprocess
from google.colab import files
nome_da_voz = re.sub(r"[^a-zA-Z0-9_-]", "_", nome_da_voz.strip()) or "minha_voz"
BRUTO = f"/content/audios/{nome_da_voz}/bruto"
DATASET = f"/content/audios/{nome_da_voz}/dataset"
os.makedirs(BRUTO, exist_ok=True)
print("Escolha um ou mais arquivos de áudio ou vídeo (MP3, WAV, M4A, MP4, OGG…):")
enviados = files.upload()
for nome, dados in enviados.items():
    with open(os.path.join(BRUTO, nome), "wb") as f:
        f.write(dados)
total = 0.0
for nome in sorted(os.listdir(BRUTO)):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", os.path.join(BRUTO, nome)], capture_output=True, text=True)
    try:
        d = float(r.stdout.strip()); total += d; print(f"  {nome}: {d/60:.1f} min")
    except ValueError:
        print(f"  {nome}: ⚠️ não consegui ler esse arquivo")
print(f"Total: {total/60:.1f} minutos de áudio")
if total < 180: print("⚠️ Pouco áudio. Funciona, mas com 10+ minutos fica bem melhor.")

Escolha um ou mais arquivos de áudio ou vídeo (MP3, WAV, M4A, MP4, OGG…):


Saving Video by alexandresilveiramg.mp3 to Video by alexandresilveiramg.mp3
  Video by alexandresilveiramg.mp3: 1.4 min
Total: 1.4 minutos de áudio
⚠️ Pouco áudio. Funciona, mas com 10+ minutos fica bem melhor.


## 3. Limpar o áudio

In [4]:
# @title Converter (e separar a voz se tiver música/barulho)
# @markdown Marque se a gravação tem música, plateia ou barulho de fundo (demora mais uns minutos).
separar_voz = False  # @param {type:"boolean"}
import os, shutil, subprocess, glob
shutil.rmtree(DATASET, ignore_errors=True); os.makedirs(DATASET)
fontes = sorted(glob.glob(f"{BRUTO}/*"))
if separar_voz:
    SEP = "/content/separador"
    if not os.path.exists(f"{SEP}/bin/audio-separator"):
        !{UV} venv {SEP} --python 3.12 --seed -q
        !{UV} pip install -q "audio-separator[gpu]==0.48.0" --python {SEP}/bin/python
    if not os.path.exists(f"{SEP}/bin/audio-separator"):
        raise SystemExit("❌ Não consegui instalar o separador. Rode este passo de novo (ou desmarque separar_voz).")
    LIMPO = f"/content/audios/{nome_da_voz}/separado"
    shutil.rmtree(LIMPO, ignore_errors=True); os.makedirs(LIMPO)
    for f in fontes:
        print("Separando:", os.path.basename(f))
        subprocess.run([f"{SEP}/bin/audio-separator", f, "--model_filename", "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
                        "--single_stem", "Vocals", "--output_format", "WAV", "--output_dir", LIMPO], check=True)
    fontes = sorted(glob.glob(f"{LIMPO}/*"))
for i, f in enumerate(fontes):
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", f, "-vn", "-ac", "1", "-ar", "40000", f"{DATASET}/{i:03d}.wav"], check=True)
print(f"✅ {len(fontes)} arquivo(s) prontos para o treino.")
if fontes:
    from IPython.display import Audio, display
    print("Ouça um pedaço pra conferir:")
    display(Audio(f"{DATASET}/000.wav"))

✅ 1 arquivo(s) prontos para o treino.
Ouça um pedaço pra conferir:


## 4. Treinar (30 min a 2 h)

In [ ]:
# @title Treinar a voz
# @markdown **300** épocas costuma ficar bom para 10–30 min de áudio. Pouco áudio (< 5 min)? Use 150–200.
epocas = 150  # @param {type:"integer"}
# @markdown Copia o resultado pro seu Google Drive conforme treina (se o Colab cair, você não perde).
salvar_no_drive = True  # @param {type:"boolean"}
import os, glob, shutil, multiprocessing
%cd /content/Applio
cpu = str(multiprocessing.cpu_count())
!{PY} core.py preprocess --model-name "{nome_da_voz}" --dataset-path "{DATASET}" --sample-rate 40000 --cpu-cores {cpu} --cut-preprocess Automatic --chunk-len 3.0 --overlap-len 0.3 --normalization-mode post
!{PY} core.py extract --model-name "{nome_da_voz}" --f0-method rmvpe --sample-rate 40000 --cpu-cores {cpu} --gpu 0 --embedder-model contentvec --include-mutes 2
!{PY} core.py train --model-name "{nome_da_voz}" --total-epoch {int(epocas)} --save-every-epoch 25 --save-only-latest --save-every-weights --sample-rate 40000 --batch-size 8 --gpu 0 --pretrained --vocoder HiFi-GAN
pesos = sorted(glob.glob(f"{APPLIO}/logs/{nome_da_voz}/{nome_da_voz}_*e_*s.pth"), key=os.path.getmtime)
if not pesos:
    raise SystemExit("❌ O treino não gerou o arquivo. Veja as mensagens acima.")
FINAL = f"/content/{nome_da_voz}.pth"
shutil.copy(pesos[-1], FINAL)
print("✅ Voz treinada:", os.path.basename(pesos[-1]))
if salvar_no_drive:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs("/content/drive/MyDrive/Vozes Caçamba", exist_ok=True)
        shutil.copy(FINAL, "/content/drive/MyDrive/Vozes Caçamba/")
        print("💾 Cópia no Google Drive: Meu Drive / Vozes Caçamba")
    except Exception as e:
        print("Não salvei no Drive:", e)

/content/Applio
Starting preprocess with 2 processes...

Preprocess completed in 1.11 seconds on 00:01:26 seconds of audio.
Model Lula preprocessed successfully.
Model Lula preprocessed successfully.
Starting pitch extraction on cuda:0 using rmvpe...

Pitch extraction completed in 5.12 seconds (32/32 files successful).
Starting embedding extraction with 2 cores on cuda:0...


Embedding extraction completed in 9.26 seconds.
Model Lula extracted successfully.
Model Lula extracted successfully.
/content/Applio/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Dataset has 1

## 5. Testar e baixar

In [ ]:
# @title Ouvir um teste e baixar o .pth
# @markdown Testa a voz em cima do primeiro áudio que você mandou (ajuste o tom se for homem → mulher, +12; mulher → homem, -12).
tom = 0  # @param {type:"slider", min:-24, max:24, step:1}
import os, subprocess
from IPython.display import Audio, display
from google.colab import files
%cd /content/Applio
teste_in = f"{DATASET}/000.wav"
teste_out = "/content/teste_voz.wav"
import glob
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", teste_in, "-t", "20", "/content/teste_in.wav"], check=True)
indices = glob.glob(f"{APPLIO}/logs/{nome_da_voz}/*.index")
indice, taxa = (indices[0], "0.5") if indices else ("", "0")  # o app não usa índice; aqui é só pro teste
!{PY} core.py infer --pitch {int(tom)} --f0-method rmvpe --input-path /content/teste_in.wav --output-path {teste_out} --pth-path "{FINAL}" --index-path "{indice}" --index-rate {taxa} --export-format WAV --embedder-model contentvec
if os.path.exists(teste_out):
    display(Audio(teste_out))
files.download(FINAL)
print("⬇️ Baixando", os.path.basename(FINAL), "— abra o Estúdio de Vozes → Carregar modelo.")

## Dicas
- **Ficou robótico / chiado?** Mais áudio limpo ajuda mais do que mais épocas. Separe a voz no passo 3.
- **Ficou abafado ou com sotaque estranho?** Treine com mais épocas (400–500) a partir do passo 4.
- **O Colab desconectou?** Rode o passo 1 de novo e depois os outros. Com *salvar no Drive* marcado, o último `.pth` fica em *Meu Drive / Vozes Caçamba*.